In [ ]:
# 1- Initial Imports
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
# 2- Dataset Initialization Phase

import os  # for file path operations
from torch.utils.data import Dataset, DataLoader  # PyTorch dataset and dataloader
from PIL import Image  # for loading images
import numpy as np  # for numerical operations
import torch  # PyTorch
import torchvision.transforms as transforms  # image preprocessing transforms
import torchvision.transforms.functional as TF  # functional transforms for augmentation
import random  # random operations for augmentation
from natsort import natsorted  # natural sorting for filenames
import platform  # to detect operating system

class SegmentationDataset(Dataset):
    def __init__(self, dataset_type, num_classes=None, augment=False):
        
        base_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices"  # base dataset directory

        # ensure dataset type and structure is valid
        if dataset_type not in ["train", "val", "test"]:
            raise ValueError("dataset_type must be 'train', 'val', or 'test'")

        # paths to images and masks
        self.image_dir = os.path.join(base_dir, dataset_type, "image")
        self.mask_dir  = os.path.join(base_dir, dataset_type, "mask")

        # load and sort filenames
        self.image_filenames = natsorted(os.listdir(self.image_dir))
        self.mask_filenames  = natsorted(os.listdir(self.mask_dir))

        # safety check
        if len(self.image_filenames) != len(self.mask_filenames):
            raise ValueError("Number of images and masks does not match")

        self.num_classes = num_classes  # number of segmentation classes
        self.augment = augment  # enable or disable augmentation

        # ImageNet normalization (provides better performance instead of the dataset's mean and standard deviation)
        dataset_mean = [0.485, 0.456, 0.406]
        dataset_std  = [0.229, 0.224, 0.225]

        # preprocessing transform for images
        self.transform_image = transforms.Compose([
            transforms.ToTensor(),  # convert PIL image to tensor
            transforms.Normalize(mean=dataset_mean, std=dataset_std)  # normalize channels
        ])

    def __len__(self):
        return len(self.image_filenames)  # return dataset size

    def __getitem__(self, idx):

        filename = self.image_filenames[idx]  # current filename

        img_path  = os.path.join(self.image_dir, self.image_filenames[idx])  # image path
        mask_path = os.path.join(self.mask_dir,  self.mask_filenames[idx])  # mask path

        # Load image (RGB)
        image = Image.open(img_path).convert("RGB")

        # Load mask (single-band TIFF with class indices)
        mask = Image.open(mask_path)
        mask_np = np.array(mask)  # convert mask to numpy array

        # Ensuring mask is single band
        if mask_np.ndim != 2:
            raise ValueError(f"Mask must be single band. Got shape: {mask_np.shape}")

        # Data augmentation (must apply same transform to image and mask)
        if self.augment:
            if random.random() > 0.5:
                image = TF.hflip(image)  # horizontal flip
                mask_np = np.fliplr(mask_np)

            if random.random() > 0.5:
                image = TF.vflip(image)  # vertical flip
                mask_np = np.flipud(mask_np)

        # Transform image
        image = self.transform_image(image)  # apply preprocessing

        # Convert mask to tensor (class indices)
        mask_tensor = torch.from_numpy(mask_np.copy()).long()

        # Safety check for class ID range (checking mask to have valid class labels)
        if self.num_classes is not None:
            if mask_tensor.max() >= self.num_classes or mask_tensor.min() < 0:
                raise ValueError(
                    f"Mask contains invalid class IDs. "
                    f"Found range [{mask_tensor.min()}, {mask_tensor.max()}], "
                    f"expected [0, {self.num_classes-1}]"
                )

        return image, mask_tensor, filename  # return image, mask, and filename

# Safe entry point for Windows multiprocessing - this is to deal with Windows11's multiprocessing error
if __name__ == "__main__":
    num_classes = 3  # number of segmentation classes

    dataset = SegmentationDataset(
        dataset_type="train",
        num_classes=num_classes,
        augment=True
    )

    num_workers = 0 if platform.system() == "Windows" else 4  # worker setting for OS

    train_loader = DataLoader(
        dataset,
        batch_size=4,  # number of samples per batch
        shuffle=True,  # shuffle training data
        num_workers=num_workers
    )

    # Test one batch
    images, masks, filenames = next(iter(train_loader))
    print("Image batch shape:", images.shape)   # [B, 3, H, W]
    print("Mask batch shape:", masks.shape)     # [B, H, W]
    print("Classes in batch:", torch.unique(masks))  # unique class IDs in batch

In [ ]:
# 3- Preparing the datasets and dataloaders
from torch.utils.data import DataLoader  # PyTorch dataloader
import platform  # detect operating system

if __name__ == "__main__":

    # initialize datasets
    train_dataset = SegmentationDataset(dataset_type="train", num_classes=num_classes, augment=True)
    val_dataset = SegmentationDataset(dataset_type="val", num_classes=num_classes, augment=False)
    test_dataset = SegmentationDataset(dataset_type="test", num_classes=num_classes, augment=False)

    # set number of workers depending on OS
    num_workers = 0 if platform.system() == "Windows" else 4

    # create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)  # training loader
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=num_workers)  # validation loader
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)  # test loader

In [ ]:
# 4- DeepLabV3 segmentation model with a custom backbone (change backbone based on your needs)

import torch  # PyTorch
import torch.nn as nn  # for building neural network modules
from torchvision.models.segmentation import DeepLabV3  # DeepLabV3 segmentation model
from torchvision.models.segmentation.deeplabv3 import DeepLabHead  # classifier head for DeepLabV3
from torchvision.models import resnet101  # ResNet101 backbone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # select GPU if available
num_classes = 3  # number of segmentation classes


class ResNet101BackboneWrapper(nn.Module):
    def __init__(self):
        super().__init__()

        # load pretrained ResNet101 backbone
        base_model = resnet101(
            weights="IMAGENET1K_V1",
            replace_stride_with_dilation=[False, True, True]  # standard for DeepLab backbones
        )

        # Remove avgpool and fc layers because we do not need classifications, we are doing segmentations
        self.backbone = nn.Sequential(*list(base_model.children())[:-2])

    def forward(self, x):
        x = self.backbone(x)  # extract feature maps
        return {"out": x}  # format expected by DeepLabV3


# Define model
backbone = ResNet101BackboneWrapper()  # initialize backbone

model = DeepLabV3(
    backbone=backbone,
    classifier=DeepLabHead(2048, num_classes)  # segmentation classifier
)

model = model.to(device)  # move model to device

print(model)  # print model architecture

In [ ]:
# 5- Setup of loss function, optimizer, and learning rate scheduler
from torch import nn, optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Initial learning rate

# StepLR scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  #dropping the learning rate by 0.1 after the step size of 3 epochs

In [ ]:
# 6- Setup of Training Function
import torch  # PyTorch
import torch.nn.functional as F  # functional neural network operations
import numpy as np  # numerical operations
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score  # evaluation metrics

def train_model(train_loader, val_loader, model, criterion, optimizer,
                scheduler=None, num_epochs=1, patience=20, ignore_class=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # select GPU if available
    model.to(device)  # move model to device

    num_classes = 3  # number of segmentation classes

    # === IOU FUNCTION ===
    def compute_iou(preds, labels, num_classes, ignore_class=ignore_class):
        preds = preds.view(-1)  # flatten predictions
        labels = labels.view(-1)  # flatten labels
        ious = []
        for cls in range(num_classes):
            if cls == ignore_class:  # skip ignored class
                continue
            pred_inds = preds == cls
            label_inds = labels == cls
            intersection = (pred_inds & label_inds).sum().item()  # number of predicted true pixels
            union = pred_inds.sum().item() + label_inds.sum().item() - intersection  # union of predicted and true pixels
            if union == 0:
                ious.append(float("nan"))
            else:
                ious.append(intersection / union)
        return ious

    best_val_loss = float("inf")  # best validation loss tracker
    epochs_no_improve = 0  # early stopping counter

    for epoch in range(num_epochs):
        # === TRAIN Function ===
        model.train()  # set model to training mode
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels, _ in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()  # reset gradients
            outputs = model(images)["out"]  # forward pass
            loss = criterion(outputs, labels)  # compute loss
            loss.backward()  # backpropagation
            optimizer.step()  # update weights

            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)  # predicted class per pixel
            correct_train += (preds == labels).sum().item()
            total_train += labels.numel()

        train_accuracy = 100 * correct_train / total_train  # training accuracy
        avg_train_loss = train_loss / len(train_loader)  # average training loss

        # === VALIDATION ===
        model.eval()  # evaluation mode
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        val_ious = []
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():  # disable gradient computation
            for images, labels, _ in val_loader:  # unpack image, mask, filename
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)["out"]  # forward pass
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.numel()

                val_ious.append(
                    compute_iou(preds, labels, num_classes, ignore_class=ignore_class)
                )

                all_val_preds.append(preds.cpu().numpy().flatten())
                all_val_labels.append(labels.cpu().numpy().flatten())

        val_accuracy = 100 * correct_val / total_val  # validation accuracy
        avg_val_loss = val_loss / len(val_loader)  # average validation loss

        # === METRICS ===
        val_ious = np.array(val_ious)
        mean_iou_per_class = np.nanmean(val_ious, axis=0)  # IoU per class
        mean_iou = np.nanmean(mean_iou_per_class)  # mean IoU

        all_val_preds = np.concatenate(all_val_preds)
        all_val_labels = np.concatenate(all_val_labels)

        conf_matrix = confusion_matrix(all_val_labels, all_val_preds)  # confusion matrix
        f1 = f1_score(all_val_labels, all_val_preds, average=None)  # F1 score per class
        precision = precision_score(all_val_labels, all_val_preds, average=None, zero_division=0)  # precision per class
        recall = recall_score(all_val_labels, all_val_preds, average=None, zero_division=0)  # recall per class

        # ==== PRINT ===
        print(
            f"\nEpoch [{epoch+1}/{num_epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.2f}% | "
            f"Val mIoU (all classes): {mean_iou:.4f}"
        )

        print("IoU per class:")
        for i, iou in enumerate(mean_iou_per_class):
            print(f"  Class {i}: {iou:.4f}")

        print("F1 score per class:")
        for i, score in enumerate(f1):
            print(f"  Class {i}: {score:.4f}")

        print("Precision per class:")
        for i, score in enumerate(precision):
            print(f"  Class {i}: {score:.4f}")

        print("Recall per class:")
        for i, score in enumerate(recall):
            print(f"  Class {i}: {score:.4f}")

        print("Confusion Matrix:")
        print(conf_matrix)

        # === SCHEDULER, CHECKPOINT & EARLY STOP ===
        # This part ensures the best model in terms of validation loss is saved
        if scheduler is not None:
            scheduler.step()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
            }, "best_checkpoint.pth")

            print(" Checkpoint saved (best validation loss)")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    return model

In [ ]:
# 7- testing, evaluation, and saving predictions

import os  # file operations
import numpy as np  # numerical operations
import torch  # PyTorch
import rasterio  # read/write TIFF files
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score  # evaluation metrics

# === SETTINGS ===
output_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices\test\predictions"
os.makedirs(output_dir, exist_ok=True)  # create output folder if it doesn't exist

num_classes = 3  # number of segmentation classes
ignore_class = None  # class to ignore in metrics

# ================= IOU FUNCTION =================
def compute_iou(all_preds, all_labels, num_classes, ignore_class=None):
    ious = []
    for cls in range(num_classes):
        if ignore_class is not None and cls == ignore_class:  # skip ignored class
            continue
        intersection = np.logical_and(all_preds == cls, all_labels == cls).sum()  # pixels correctly predicted
        union = np.logical_or(all_preds == cls, all_labels == cls).sum()  # total pixels predicted or true
        if union == 0:
            ious.append(np.nan)
        else:
            ious.append(intersection / union)  # IoU for class
    return np.array(ious)

# === TEST FUNCTION ===
def test_model(test_loader, model, criterion):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # select GPU if available
    model.to(device)
    model.eval()  # set model to evaluation mode
    running_loss = 0.0  # cumulative loss
    total_correct = 0  # correct pixels
    total_pixels = 0  # total pixels
    all_preds = []  # store predictions
    all_labels = []  # store true labels

    with torch.no_grad():  # disable gradient computation
        for images, labels, filenames in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)["out"]  # forward pass
            loss = criterion(outputs, labels)
            running_loss += loss.item()  # accumulate loss
            predicted = torch.argmax(outputs, dim=1)  # predicted class per pixel

            # ================= SAVE PREDICTIONS =================
            for i in range(predicted.shape[0]):
                pred_mask = predicted[i].cpu().numpy().astype(np.uint8)  # convert to numpy uint8
                base_filename = os.path.splitext(filenames[i])[0]
                tif_path = os.path.join(output_dir, base_filename + ".tif")
                with rasterio.open(
                    tif_path,
                    "w",
                    driver="GTiff",
                    height=pred_mask.shape[0],
                    width=pred_mask.shape[1],
                    count=1,
                    dtype="uint8"
                ) as dst:
                    dst.write(pred_mask, 1)  # write mask to TIFF

            # === ACCURACY (For All Pixels) ===
            correct = (predicted == labels).sum().item()
            total = labels.numel()
            total_correct += correct
            total_pixels += total
            all_preds.append(predicted.cpu().numpy().flatten())  # flatten for metrics
            all_labels.append(labels.cpu().numpy().flatten())

    # === FINAL METRICS ===
    avg_loss = running_loss / len(test_loader)  # average test loss
    avg_accuracy = total_correct / total_pixels  # global pixel accuracy
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    conf_matrix = confusion_matrix(all_labels, all_preds, labels=[0,1,2])
    f1 = f1_score(all_labels, all_preds, labels=[0,1,2], average=None)
    precision = precision_score(all_labels, all_preds, labels=[0,1,2], average=None, zero_division=0)
    recall = recall_score(all_labels, all_preds, labels=[0,1,2], average=None, zero_division=0)
    ious = compute_iou(all_preds, all_labels, num_classes, ignore_class)
    miou = np.nanmean(ious)  # mean IoU

    producer_accuracy = np.diag(conf_matrix) / np.sum(conf_matrix, axis=1)  # row-wise accuracy
    user_accuracy = np.diag(conf_matrix) / np.sum(conf_matrix, axis=0)  # column-wise accuracy

    # === PRINT RESULTS ===
    print(f"\nTest Loss: {avg_loss:.4f}")
    print(f"Global Pixel Accuracy: {avg_accuracy:.4f}")
    print(f"Mean IoU: {miou:.4f}")

    print("\nIoU per class:")
    for i, iou in enumerate(ious):
        print(f"  Class {i}: {iou:.4f}")

    print("\nF1 Score per class:")
    for i, score in enumerate(f1):
        print(f"  Class {i}: {score:.4f}")

    print("\nPrecision per class:")
    for i, score in enumerate(precision):
        print(f"  Class {i}: {score:.4f}")

    print("\nRecall per class:")
    for i, score in enumerate(recall):
        print(f"  Class {i}: {score:.4f}")

    print("\nConfusion Matrix:")
    print(conf_matrix)

    print("\nProducer Accuracy per class:")
    print(producer_accuracy)

    print("\nUser Accuracy per class:")
    print(user_accuracy)

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "iou": ious,
        "miou": miou,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "confusion_matrix": conf_matrix,
        "producer_accuracy": producer_accuracy,
        "user_accuracy": user_accuracy
    }

In [ ]:
# 8- Training Phase

if __name__ == "__main__":
    num_epochs = 5
    patience = 20
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # === TRAIN ===
    trained_model = train_model(
        train_loader=train_loader,
        val_loader=val_loader,
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        patience=patience
    )

    # === LOAD BEST CHECKPOINT ===
    checkpoint = torch.load("best_checkpoint.pth", map_location=device)
    trained_model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded best validation checkpoint") 
    
    # === SAVE FINAL MODEL ===
    model_save_path = r"C:/My Files/Semester4TUM/Developed Dataset/Prepared Dataset for Thesis - fixed class imbalance/forthandback.pth"
    torch.save(trained_model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")

In [ ]:
# 9- Calling the test function after the training is complete

test_metrics = test_model(
        test_loader,
        trained_model,
        criterion
     )
print("\nTest Accuracy (overall pixel accuracy):", test_metrics["accuracy"])

In [ ]:
#10 Loading The Model Back and setting it back to evaluation mode

backbone = ResNet101BackboneWrapper()

model = DeepLabV3(
    backbone=backbone,
    classifier=DeepLabHead(2048, num_classes)
)

model.load_state_dict(torch.load(model_save_path))

model = model.to(device)

model.eval()  # set to evaluation mode

In [ ]:
# 11- Inference on big tile for city-wide inference

import rasterio  # for reading large raster/TIFF files
import numpy as np  # for numerical operations
from PIL import Image  # for converting numpy arrays to PIL images
import torchvision.transforms as T  # for preprocessing transforms
import torch  # for PyTorch model inference

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # select GPU if available

# Patch size
patch_height = 224  # height of each sliding window patch
patch_width = 224  # width of each patch

# Stride
stride_y = patch_height // 2  # vertical stride: 50% overlap to reduce artifacts
stride_x = patch_width // 2   # horizontal stride: 50% overlap

# Batch Size
batch_size = 8  # number of patches processed together

# Path to large TIFF image
tiff_path = r"C:\My Files\Semester4TUM\2017 Image Resampled\R3C6.tif"

# Load full resolution image
with rasterio.open(tiff_path) as src:
    img_np = src.read()  # read image as (bands, height, width)
    img_np = np.transpose(img_np, (1, 2, 0))  # reorder to (height, width, channels)
    img_np = np.clip(img_np, 0, 255).astype(np.uint8)  # clip to 0-255 and convert to uint8

# Transform - should be same as the training phase
transform = T.Compose([
    T.ToTensor(),  # convert PIL image to PyTorch tensor (C, H, W) and scale to [0,1]
    T.Normalize(mean=[0.485, 0.456, 0.406],  # normalize channels with ImageNet mean
                std=[0.229, 0.224, 0.225]),  # normalize channels with ImageNet std
])

# Get image dimensions
H, W, C = img_np.shape

# Calculate number of patches in y and x directions
n_patches_y = (H - patch_height) // stride_y + 1
n_patches_x = (W - patch_width) // stride_x + 1

num_classes = 3  # number of classes in segmentation
# prepare empty arrays to accumulate patch predictions
output_probs = np.zeros((num_classes, H, W), dtype=np.float32)  
count_mask = np.zeros((H, W), dtype=np.float32)  # to count overlapping patch contributions

model.eval()  # set model to evaluation mode

with torch.no_grad():  # disable gradient computation
    patches = []  # list to store current batch of patches
    coords = []   # list to store coordinates of patches

    # loop over patches in vertical direction
    for i in range(n_patches_y):
        # loop over patches in horizontal direction
        for j in range(n_patches_x):
            y = i * stride_y  # top coordinate of current patch
            x = j * stride_x  # left coordinate of current patch

            patch = img_np[y:y+patch_height, x:x+patch_width, :]  # crop patch from large image
            patch_pil = Image.fromarray(patch)  # convert patch to PIL image
            input_tensor = transform(patch_pil)  # apply preprocessing transforms

            patches.append(input_tensor)  # add to current batch
            coords.append((y, x))  # store coordinates

            # Process batch when enough patches collected
            if len(patches) == batch_size:
                batch_tensor = torch.stack(patches).to(device)  # stack and move batch to device
                outputs = model(batch_tensor)['out']  # forward pass, get output probabilities
                outputs_np = outputs.cpu().numpy()  # move to CPU and convert to numpy

                # Add predictions to the large output array
                for idx, (yy, xx) in enumerate(coords):
                    output_probs[:, yy:yy+patch_height, xx:xx+patch_width] += outputs_np[idx]
                    count_mask[yy:yy+patch_height, xx:xx+patch_width] += 1  # increment overlap counter

                patches = []  # reset batch
                coords = []   # reset coordinates

    # Process remaining patches if batch not full
    if len(patches) > 0:
        batch_tensor = torch.stack(patches).to(device)
        outputs = model(batch_tensor)['out']
        outputs_np = outputs.cpu().numpy()

        # Add remaining predictions to the large output array
        for idx, (yy, xx) in enumerate(coords):  # stitching back the patches
            output_probs[:, yy:yy+patch_height, xx:xx+patch_width] += outputs_np[idx]
            count_mask[yy:yy+patch_height, xx:xx+patch_width] += 1

# Average overlapping predictions to normalize
output_probs /= count_mask[np.newaxis, :, :]

# Final predicted mask: choose class with highest probability at each pixel
predicted_mask = np.argmax(output_probs, axis=0)

print("Prediction done. Final mask shape:", predicted_mask.shape)

In [ ]:
#12 works Full mask visualization   
import numpy as np
import matplotlib.pyplot as plt

# Define the colormap array, index = class id
colormap = np.array([
    [74, 100, 145],       # Class 0 Formal
    [224, 224, 224],     # Class 1 Background
    [217, 95, 14],     # Class 2 Informal
], dtype=np.uint8)

# Map predicted_mask class IDs to colors
color_mask = colormap[predicted_mask]

plt.figure(figsize=(8, 8))
plt.imshow(color_mask)
plt.title("Predicted Mask Colored")
plt.axis('off')
plt.show()


In [ ]:
#13 Save the citywide predicted mask in a singleband format
import rasterio
import numpy as np

# Check which classes are present
unique_classes = np.unique(predicted_mask)
print("Classes present in predicted mask:", unique_classes)

# Open original raster to get metadata
with rasterio.open(tiff_path) as src:
    meta = src.meta.copy()

# Update metadata for single-band
meta.update({
    "count": 1,          # single band
    "dtype": "uint8"     # class indices fit in uint8
})

# Save predicted mask as single-band GeoTIFF
output_tif = r"C:\My Files\Semester4TUM\R3C6_Predict.tif"
with rasterio.open(output_tif, "w", **meta) as dst:
    dst.write(predicted_mask.astype(np.uint8), 1)  # band 1

print("Single-band GeoTIFF with class indices saved:", output_tif)